# 02 -- model training (v2)

trains the neural network board evaluator on the 47-million sample augmented dataset
generated in notebook 01 (`training_data_v2.npz`).

architecture:
- input: 16 normalized floats (log2(v) / 17)
- dense 128 relu
- dense 64 relu
- dense 1 linear (predicted score-to-end / score_max)

output:
- `model.json` and `group1-shard1of1.bin` (tensorflow.js format)
- `score_max.npy` (max score scalar for web app denormalization)
- packaged in `02_model_training_output.zip`

In [1]:
import os
import shutil
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'tensorflow version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'detected gpus: {gpus}')
if gpus:
    print('running on gpu accelerator!')
else:
    print('warning: no gpu detected, training on cpu')

tensorflow version: 2.20.0
detected gpus: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
running on gpu accelerator!


## load and prepare dataset

In [2]:
# find dataset file (local or kaggle input dataset path)
possible_paths = [
    'training_data_v2.npz',
    'training_data.npz',
    '../input/2048-training-dataset/training_data_v2.npz',
    '../input/2048-training-dataset/training_data.npz'
]

# search /kaggle/input if running on kaggle
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file.endswith('.npz'):
                possible_paths.insert(0, os.path.join(root, file))

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if not data_path:
    raise FileNotFoundError('could not find training_data_v2.npz or training_data.npz')

print(f'loading dataset from: {data_path}')
data = np.load(data_path)
boards = data['boards']
scores = data['scores']

print(f'total samples: {len(boards):,}')
print(f'boards shape:  {boards.shape}')
print(f'scores shape:  {scores.shape}')
print(f'score min:     {scores.min():.0f}')
print(f'score max:     {scores.max():.0f}')

# save and record score_max for denormalization
score_max = float(scores.max())
np.save('score_max.npy', np.array(score_max, dtype=np.float32))
print(f'recorded score_max: {score_max:.0f}')

# normalize scores to [0.0, 1.0]
scores_norm = (scores / score_max).astype(np.float32)

# 90% train, 10% validation split
n = len(boards)
indices = np.random.permutation(n)
split = int(n * 0.9)

x_train = boards[indices[:split]]
y_train = scores_norm[indices[:split]]
x_val = boards[indices[split:]]
y_val = scores_norm[indices[split:]]

print(f'train set: {len(x_train):,} samples')
print(f'val set:   {len(x_val):,} samples')

loading dataset from: /kaggle/input/datasets/maheshkakde165/2048-training-dataset-v2/training_data_v2.npz
total samples: 47,003,816
boards shape:  (47003816, 16)
scores shape:  (47003816,)
score min:     0
score max:     76564
recorded score_max: 76564
train set: 42,303,434 samples
val set:   4,700,382 samples


## build model architecture

In [3]:
model = keras.Sequential([
    layers.Input(shape=(16,)),
    layers.Dense(128, activation='relu', name='dense'),
    layers.Dense(64, activation='relu', name='dense_1'),
    layers.Dense(1, activation='linear', name='dense_2')
], name='sequential')

# use learning rate with warmup/decay for high stability on 47m samples
optimizer = keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

model.summary()

I0000 00:00:1791899991.897224      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791899991.901192      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,497 (41.00 KB)

 Trainable params: 10,497 (41.00 KB)

 Non-trainable params: 0 (0.00 B)

## train model on 47m samples

In [4]:
BATCH_SIZE = 4096
EPOCHS = 15

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

print(f'starting training: batch_size={BATCH_SIZE}, epochs={EPOCHS}...')
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

final_val_loss = min(history.history['val_loss'])
final_val_mae = min(history.history['val_mae'])
print(f'training complete!')
print(f'best val mse loss: {final_val_loss:.6f}')
print(f'best val mae:      {final_val_mae:.6f} ({final_val_mae * score_max:.1f} score points)')

starting training: batch_size=4096, epochs=15...
Epoch 1/15
   63/10328 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step - loss: 0.0242 - mae: 0.1224

I0000 00:00:1791900006.253463     141 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10328/10328 ━━━━━━━━━━━━━━━━━━━━ 36s 3ms/step - loss: 0.0148 - mae: 0.0968 - val_loss: 0.0147 - val_mae: 0.0967 - learning_rate: 0.0010
Epoch 2/15
10328/10328 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 0.0147 - mae: 0.0959 - val_loss: 0.0147 - val_mae: 0.0954 - learning_rate: 0.0010
Epoch 3/15
10325/10328 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0147 - mae: 0.0957
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
10328/10328 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 0.0146 - mae: 0.0957 - val_loss: 0.0147 - val_mae: 0.0954 - learning_rate: 0.0010
Epoch 4/15
10328/10328 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 0.0146 - mae: 0.0955 - val_loss: 0.0146 - val_mae: 0.0957 - learning_rate: 5.0000e-04
Epoch 5/15
10328/10328 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 0.0146 - mae: 0.0955 - val_loss: 0.0146 - val_mae: 0.0954 - learning_rate: 5.0000e-04
Epoch 6/15
10317/10328 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0146 - mae: 0.0955
Epoch 6: ReduceLROnPlateau reduci

## evaluate on validation set

In [5]:
val_preds = model.predict(x_val[:10000], batch_size=4096).flatten()
actual_scores = y_val[:10000] * score_max
predicted_scores = val_preds * score_max

errors = np.abs(actual_scores - predicted_scores)
print(f'sample validation evaluation (10,000 states):')
print(f'  mean absolute error: {errors.mean():.1f} points')
print(f'  median absolute error: {np.median(errors):.1f} points')
print(f'  max error:             {errors.max():.1f} points')

# print a few sample comparisons
print('\nsample predictions vs actual values:')
for i in range(5):
    print(f'  sample {i+1}: predicted={predicted_scores[i]:.0f} | actual={actual_scores[i]:.0f} | diff={errors[i]:.0f}')

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
sample validation evaluation (10,000 states):
  mean absolute error: 7335.5 points
  median absolute error: 6199.3 points
  max error:             52469.2 points

sample predictions vs actual values:
  sample 1: predicted=12106 | actual=12564 | diff=458
  sample 2: predicted=17273 | actual=11156 | diff=6117
  sample 3: predicted=19104 | actual=13208 | diff=5896
  sample 4: predicted=10502 | actual=4156 | diff=6346
  sample 5: predicted=19043 | actual=13468 | diff=5575


## export to tensorflow.js format

In [6]:
def export_to_tfjs(model, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    bin_filename = 'group1-shard1of1.bin'
    bin_path = os.path.join(output_dir, bin_filename)
    
    weight_entries = []
    all_weight_bytes = bytearray()
    
    for weight in model.weights:
        arr = weight.numpy().astype(np.float32)
        weight_entries.append({
            'name': weight.name,
            'shape': list(arr.shape),
            'dtype': 'float32'
        })
        all_weight_bytes.extend(arr.tobytes())
    
    with open(bin_path, 'wb') as f:
        f.write(all_weight_bytes)
        
    model_json = {
        'format': 'layers-model',
        'generatedBy': 'keras v3',
        'convertedBy': 'TensorFlow.js Converter',
        'modelTopology': {
            'class_name': 'Sequential',
            'config': {
                'name': 'sequential',
                'layers': [
                    {
                        'class_name': 'InputLayer',
                        'config': {
                            'batch_input_shape': [None, 16],
                            'dtype': 'float32',
                            'sparse': False,
                            'name': 'dense_input'
                        }
                    },
                    {
                        'class_name': 'Dense',
                        'config': {
                            'name': 'dense',
                            'trainable': True,
                            'dtype': 'float32',
                            'units': 128,
                            'activation': 'relu',
                            'use_bias': True
                        }
                    },
                    {
                        'class_name': 'Dense',
                        'config': {
                            'name': 'dense_1',
                            'trainable': True,
                            'dtype': 'float32',
                            'units': 64,
                            'activation': 'relu',
                            'use_bias': True
                        }
                    },
                    {
                        'class_name': 'Dense',
                        'config': {
                            'name': 'dense_2',
                            'trainable': True,
                            'dtype': 'float32',
                            'units': 1,
                            'activation': 'linear',
                            'use_bias': True
                        }
                    }
                ]
            },
            'keras_version': '2.15.0',
            'backend': 'tensorflow'
        },
        'weightsManifest': [
            {
                'paths': [bin_filename],
                'weights': weight_entries
            }
        ]
    }
    
    with open(os.path.join(output_dir, 'model.json'), 'w', encoding='utf-8') as f:
        json.dump(model_json, f, indent=2)

export_to_tfjs(model, 'tfjs_model')
print('successfully exported to tfjs_model/')
for f in os.listdir('tfjs_model'):
    size = os.path.getsize(os.path.join('tfjs_model', f))
    print(f'  {f} ({size} bytes)')

successfully exported to tfjs_model/
  group1-shard1of1.bin (41988 bytes)
  model.json (2309 bytes)


## package output for download

In [7]:
os.makedirs('output', exist_ok=True)

# copy tfjs model files
for f in os.listdir('tfjs_model'):
    shutil.copy(os.path.join('tfjs_model', f), os.path.join('output', f))

# copy score_max
shutil.copy('score_max.npy', 'output/score_max.npy')

shutil.make_archive('02_model_training_output', 'zip', 'output')
print('done. download 02_model_training_output.zip')
print('\npackage contents:')
for f in os.listdir('output'):
    print(f'  {f} ({os.path.getsize(os.path.join("output", f))} bytes)')

done. download 02_model_training_output.zip

package contents:
  group1-shard1of1.bin (41988 bytes)
  score_max.npy (132 bytes)
  model.json (2309 bytes)
